# InsuranceQA v2: Gemma synthetic-data generator

**Kaggle setup:** GPU **T4 x2**, Internet **On**, Python 3.10–3.13. Run this notebook separately from the other two.
This notebook contains every source file it uses. It trains its own independent QLoRA adapter, measures distribution
fidelity, and exports a portable adapter package. No training or quality results are pre-filled.

**Before running:** read the data-audit notes below. The base model is Gemma 4 E4B; the visible generation prompt is
`Generate synthetic data.` Internal schema instructions and jointly sampled training attributes provide control.
The ten-hour work budget starts at the pipeline cell; environment installation is additional session time.
Do not publish generated outputs merely because JSON validation passes. Read the report and complete the review sample.


## Data audit

The verified source contains 12,889/2,000/2,000 train/validation/test questions. Near-duplicate question isolation retains 12,856/1,998/2,000. Shared professional answers are counted separately. The source describes research-only use.

The pipeline recomputes counts from pinned source files and saves every exclusion in `split_manifest.jsonl`. MIMIC is outside this notebook.


## Configuration

Change the runtime budget or sample counts here. A larger sample improves estimates but increases generation time. Maximum epochs are ceilings, not a promise that the session can finish every epoch.


In [1]:
import os, sys, json, time, subprocess, shutil
from pathlib import Path
assert sys.platform.startswith("linux"), "Run this notebook in Kaggle's Linux GPU environment."
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Use Python 3.11–3.13 (trl 1.15 requires >=3.11)."
PROGRAM = Path("/kaggle/working/insurance_gemma_program")
(PROGRAM / "src").mkdir(parents=True, exist_ok=True)
# ---- Safety settings -------------------------------------------------------------------------------
# BACKUP: pushes checkpoints/reports to a PRIVATE Hugging Face repo every BACKUP_EVERY_MIN minutes so a killed or
# quota-exhausted Kaggle session cannot destroy hours of work. Needs a Kaggle secret HF_TOKEN with WRITE access.
BACKUP_REQUIRED = True      # True: stop immediately (before any GPU time) if backup cannot be written
BACKUP_EVERY_MIN = 30
# RESUME after a killed session: set RESUME_RUN_NAME to the old run folder name (e.g. 'insuranceqa_20261101_093000') and
# RESUME_SOURCE to either "hf" (pull from the backup repo) or a path to the old run folder (e.g. an attached
# previous notebook output under /kaggle/input/...). Leave both None for a fresh run.
RESUME_RUN_NAME = None
RESUME_SOURCE = None
RUN_NAME = RESUME_RUN_NAME or ('insuranceqa' + "_" + time.strftime("%Y%m%d_%H%M%S"))
RUN_DIR = Path("/kaggle/working") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
HF_TOKEN_PRESENT = False
CONFIG = {'dataset': 'insuranceqa', 'dataset_revision': '5c380dd086067adacc2fec3bfa920a78e57bdcf6', 'model_id': 'google/gemma-4-E4B-it', 'model_revision': 'ee0ef6023621cff504d758262d4e04895a5af4a2', 'embedding_id': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41', 'seed': 3407, 'hours': 10.5, 'rank': 16, 'alpha': 32, 'learning_rate': 0.0001, 'epochs': 3, 'selection_n': 64, 'final_n': 128, 'bootstrap_repeats': 30, 'embedding_device': 'cpu',
          'train_eval_examples': 400, 'save_every_steps': 100,
          'train_until_done': True, 'early_stopping_patience': 0,
          'train_fraction': 0.60, 'selection_fraction': 0.80, 'final_fraction': 0.96,
          'selection_temperatures': [0.9, 0.7, 1.0], 'evaluate_base': True, 'min_partial_attempts': 16}
SAFETY_PREFLIGHT = False

CONFIG["run_dir"] = str(RUN_DIR)
CONFIG_PATH = RUN_DIR / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
os.environ["HF_HOME"] = "/tmp/insurance_gemma_hf"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
# Optional: add a Kaggle secret named HF_TOKEN if Hugging Face requests authentication.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); HF_TOKEN_PRESENT = True
except Exception:
    pass  # Public sources normally work without a token; no secrets are printed.
print("Run:", RUN_DIR)
print(json.dumps(CONFIG, indent=2))


Run: /kaggle/working/insuranceqa_20261031_194809
{
  "dataset": "insuranceqa",
  "dataset_revision": "5c380dd086067adacc2fec3bfa920a78e57bdcf6",
  "model_id": "google/gemma-4-E4B-it",
  "model_revision": "ee0ef6023621cff504d758262d4e04895a5af4a2",
  "embedding_id": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "seed": 3407,
  "hours": 10.5,
  "rank": 16,
  "alpha": 32,
  "learning_rate": 0.0001,
  "epochs": 3,
  "selection_n": 64,
  "final_n": 128,
  "bootstrap_repeats": 30,
  "embedding_device": "cpu",
  "train_eval_examples": 400,
  "save_every_steps": 100,
  "train_until_done": true,
  "early_stopping_patience": 0,
  "train_fraction": 0.6,
  "selection_fraction": 0.8,
  "final_fraction": 0.96,
  "selection_temperatures": [
    0.9,
    0.7,
    1.0
  ],
  "evaluate_base": true,
  "min_partial_attempts": 16,
  "run_dir": "/kaggle/working/insuranceqa_20261031_194809"
}


## Embedded implementation

These cells write the complete pipeline to the working directory. Expand them to inspect or edit the code.


In [2]:
%%writefile /kaggle/working/insurance_gemma_program/src/core.py
"""Deterministic, dependency-free preparation and validation primitives."""
from __future__ import annotations
import collections, hashlib, json, math, random, re
from pathlib import Path

KINDS = ('insuranceqa', 'bitext', 'reliability')
NUMBER_PATTERNS={
    'money_usd':r'\$\s*(\d[\d,]*(?:\.\d+)?)',
    'percent':r'(\d+(?:\.\d+)?)\s*%',
    'duration_days':r'\b(\d+(?:\.\d+)?)\s*days?\b',
    'duration_months':r'\b(\d+(?:\.\d+)?)\s*months?\b',
    'age_years':r'\b(\d+(?:\.\d+)?)\s*(?:years? old|year-old)\b',
}

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def read_jsonl(path):
    with Path(path).open(encoding='utf-8-sig') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl(path, rows):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for row in rows: handle.write(canonical(row) + '\n')

def normalize(text):
    return ' '.join(re.findall(r"\w+", str(text).casefold()))

def fingerprint(record):
    # IDs assigned by the application must never conceal a copied record.
    value = {k: v for k, v in record.items() if k != 'id'}
    def clean(x):
        if isinstance(x, str): return ' '.join(x.casefold().split())
        if isinstance(x, list): return [clean(v) for v in x]
        if isinstance(x, dict): return {k: clean(v) for k, v in x.items()}
        return x
    return hashlib.sha256(canonical(clean(value)).encode()).hexdigest()

def primary_text(record, kind):
    if kind == 'insuranceqa': return record['question']
    if kind == 'bitext': return record['instruction']
    return record['input']['customer_message']

def strings(value):
    if isinstance(value, str): return [value]
    if isinstance(value, dict): return [s for k,v in value.items() if k != 'id' for s in strings(v)]
    if isinstance(value, list): return [s for v in value for s in strings(v)]
    return []

def text_fields(record, kind):
    if kind == 'insuranceqa': return {'question': record['question'], 'answer': '\n'.join(record['answers'])}
    if kind == 'bitext': return {'question': record['instruction'], 'answer': record['response']}
    out = record['expected_output']
    response = next((out[k] for k in ('response', 'response_template', 'suggested_response', 'customer_response') if k in out), out)
    return {'question': record['input']['customer_message'], 'answer': '\n'.join(strings(response)),
            'context': '\n'.join(strings(record['input'].get('context', {})))}

def text(record, kind):
    return '\n'.join(text_fields(record, kind).values())

def leaves(value, prefix=''):
    if isinstance(value, dict):
        for k,v in value.items():
            yield from leaves(v, f'{prefix}.{k}' if prefix else k)
    elif not isinstance(value, list):
        yield prefix, value

def get_path(record, path, default=None):
    for key in path.split('.'):
        if not isinstance(record, dict) or key not in record: return default
        record = record[key]
    return record

def quantity_features(record,kind):
    result={}
    for field,value in text_fields(record,kind).items():
        for name,pattern in NUMBER_PATTERNS.items():
            result[f'{field}.{name}']=[float(x.replace(',','')) for x in re.findall(pattern,value,re.I)]
    for path,value in leaves(record):
        if type(value) in (int,float) and math.isfinite(value): result['structured.'+path]=[float(value)]
    return result

def condition_paths(kind, records):
    if kind == 'insuranceqa': return ['domain']
    if kind == 'bitext': return ['intent', 'category', 'tags']
    candidates = ['metadata.insurance_line', 'metadata.insurance_type', 'insurance_line']
    line = next((p for p in candidates if all(get_path(r,p) is not None for r in records)), None)
    if line is None: raise ValueError('Cannot locate insurance-line metadata; inspect the source schema.')
    return ['category', 'difficulty', 'expected_output.routing_decision', line]

def condition(record, paths):
    return {p: get_path(record,p) for p in paths}

def shingles(value):
    words = normalize(value).split()
    # Short instructions require exact normalized matching.
    return frozenset(tuple(words[i:i+3]) for i in range(len(words)-2)) if len(words)>=3 else frozenset([tuple(words)])

def jaccard(a,b):
    return len(a & b)/len(a | b) if a or b else 1.0

def duplicate_groups(texts, threshold=.85):
    """Exact prefix-filtered Jaccard join; unlike LSH, no probabilistic misses."""
    sets = [shingles(s) for s in texts]
    frequency = collections.Counter(s for tokens in sets for s in tokens)
    ordered = [sorted(s, key=lambda t:(frequency[t],t)) for s in sets]
    parent = list(range(len(sets)))
    def find(i):
        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(i,j):
        a,b=find(i),find(j)
        if a!=b: parent[max(a,b)] = min(a,b)
    index=collections.defaultdict(list)
    for i in sorted(range(len(sets)), key=lambda k:(len(sets[k]),k)):
        tokens=ordered[i]; prefix=tokens[:len(tokens)-math.ceil(threshold*len(tokens))+1]
        candidates={j for t in prefix for j in index[t] if len(sets[j]) >= threshold*len(sets[i])}
        for j in candidates:
            if jaccard(sets[i],sets[j]) >= threshold: union(i,j)
        for token in prefix: index[token].append(i)
    return [find(i) for i in range(len(texts))]

def grouped_split(records, kind, seed=3407):
    groups=duplicate_groups([primary_text(r,kind) for r in records])
    members=collections.defaultdict(list)
    for i,g in enumerate(groups): members[g].append(i)
    labels=[r['intent'] for r in records]
    totals=collections.Counter(labels); ratios={'train':.8,'validation':.1,'test':.1}
    counts={s:collections.Counter() for s in ratios}; assignment={}
    order=list(members); random.Random(seed).shuffle(order); order.sort(key=lambda g:-len(members[g]))
    for g in order:
        group_counts=collections.Counter(labels[i] for i in members[g])
        # Minimize the increase in squared stratification error; large groups stay intact.
        def cost(s):
            return sum(((counts[s][k]+v-ratios[s]*totals[k])**2-
                        (counts[s][k]-ratios[s]*totals[k])**2)/max(ratios[s]*totals[k],1)
                       for k,v in group_counts.items())
        split=min(ratios,key=cost); counts[split].update(group_counts)
        for i in members[g]: assignment[i]=split
    return assignment,groups

def audit_official(rows, kind):
    groups=duplicate_groups([primary_text(x['record'],kind) for x in rows])
    seen=collections.defaultdict(set)
    for row,g in zip(rows,groups): seen[g].add(row['split'])
    kept=[]; manifest=[]; priorities={'train':0,'validation':1,'test':2}
    for row,g in zip(rows,groups):
        owner=max(seen[g],key=priorities.get)
        keep=row['split']==owner
        manifest.append({'uid':row['uid'],'source_split':row['split'],'split':row['split'] if keep else 'excluded',
                         'group':g,'reason':None if keep else f'question_overlap_with_{owner}',
                         'sha256':fingerprint(row['record'])})
        if keep: kept.append(row)
    return kept,manifest

def infer_schema(values):
    schemas=[]
    by_type=collections.defaultdict(list)
    for v in values:
        t='null' if v is None else 'boolean' if isinstance(v,bool) else 'object' if isinstance(v,dict) else 'array' if isinstance(v,list) else 'integer' if isinstance(v,int) else 'number' if isinstance(v,float) else 'string'
        by_type[t].append(v)
    for t,vs in sorted(by_type.items()):
        schema={'type':t}
        if t=='object':
            keys=set().union(*(v.keys() for v in vs))
            schema.update(properties={k:infer_schema([v[k] for v in vs if k in v]) for k in sorted(keys)},
                          required=sorted(set.intersection(*(set(v) for v in vs))), additionalProperties=False)
        elif t=='array':
            items=[x for v in vs for x in v]
            schema['items']=infer_schema(items) if items else {}
        elif t=='string' and all(v.strip() for v in vs): schema['minLength']=1
        schemas.append(schema)
    return schemas[0] if len(schemas)==1 else {'anyOf':schemas}

def schema_errors(value, schema, path='$'):
    if not schema: return []
    if 'anyOf' in schema:
        return [] if any(not schema_errors(value,s,path) for s in schema['anyOf']) else [path+':type_union']
    t=schema['type']; checks={'object':lambda: isinstance(value,dict),'array':lambda:isinstance(value,list),
        'string':lambda:isinstance(value,str),'boolean':lambda:type(value) is bool,'integer':lambda:type(value) is int,
        'number':lambda:type(value) in (int,float) and math.isfinite(value),'null':lambda:value is None}
    if not checks[t](): return [path+':type_'+t]
    errors=[]
    if t=='object':
        errors += [path+':missing_'+k for k in schema['required'] if k not in value]
        errors += [path+':extra_'+k for k in value if k not in schema['properties']]
        for k in value:
            if k in schema['properties']: errors += schema_errors(value[k],schema['properties'][k],path+'.'+k)
    elif t=='array':
        for i,v in enumerate(value): errors += schema_errors(v,schema['items'],f'{path}[{i}]')
    elif t=='string' and schema.get('minLength') and not value.strip(): errors.append(path+':empty')
    return errors

def fit_profile(records, kind):
    if not records: raise ValueError('Empty training split after leakage filtering.')
    paths=condition_paths(kind,records)
    tuples=collections.Counter(canonical(condition(r,paths)) for r in records)
    placeholders=sorted(set(re.findall(r'\{\{[^{}]+\}\}', '\n'.join(text(r,kind) for r in records))))
    categories={}
    for r in records:
        for p,v in leaves(r):
            if isinstance(v,(str,bool)) and len(str(v))<100 and p!='id': categories.setdefault(p,set()).add(canonical(v))
    categories={p:sorted(v) for p,v in categories.items() if len(v)<=64 and p not in ['question','instruction','response','input.customer_message']}
    label_paths=paths+(['subcategory','expected_output.intent','expected_output.priority',
        'input.context.policy_type','input.context.policy_status','input.context.customer_tier',
        'input.context.time_of_day','input.context.channel','input.context.state'] if kind=='reliability' else [])
    list_paths=['expected_output.required_actions','expected_output.compliance_flags','expected_output.expected_response_elements'] if kind=='reliability' else []
    quantities=collections.defaultdict(list)
    for record in records:
        for key,values in quantity_features(record,kind).items(): quantities[key].extend(values)
    return {'kind':kind,'schema':infer_schema(records),'condition_paths':paths,'joint_counts':dict(tuples),
            'placeholders':placeholders,'categorical_paths':sorted(set(categories)|set(paths)),
            'allowed_values':{p:sorted({canonical(get_path(r,p)) for r in records}) for p in label_paths},
            'allowed_list_values':{p:sorted({canonical(x) for r in records for x in get_path(r,p,[])}) for p in list_paths},
            'numeric_bounds':{k:{'min':min(v),'max':max(v),'n':len(v)} for k,v in quantities.items() if v},
            'training_hashes':sorted({fingerprint(r) for r in records}),
            'training_count':len(records)}

def sample_conditions(profile, n, seed):
    counts=profile['joint_counts']; keys=list(counts)
    return [json.loads(k) for k in random.Random(seed).choices(keys,weights=[counts[k] for k in keys],k=n)]

def validate_record(record, profile, requested=None, reject_copy=False):
    if not isinstance(record,dict): return ['not_object']
    errors=schema_errors(record,profile['schema'])
    if errors: return errors
    if profile['kind']=='insuranceqa' and not record['answers']: errors.append('empty_answers')
    combo=condition(record,profile['condition_paths'])
    for path,allowed in profile['allowed_values'].items():
        if canonical(get_path(record,path)) not in allowed: errors.append('unknown_label:'+path)
    for path,allowed in profile.get('allowed_list_values',{}).items():
        if any(canonical(x) not in allowed for x in get_path(record,path,[])): errors.append('unknown_list_label:'+path)
    if canonical(combo) not in profile['joint_counts']: errors.append('unseen_condition_combination')
    if requested is not None and combo!=requested: errors.append('condition_mismatch')
    content=text(record,profile['kind'])
    ph=re.findall(r'\{\{[^{}]+\}\}',content)
    if set(ph)-set(profile['placeholders']): errors.append('unknown_placeholder')
    stripped=re.sub(r'\{\{[^{}]+\}\}','',content)
    if '{{' in stripped or '}}' in stripped: errors.append('malformed_placeholder')
    if reject_copy and fingerprint(record) in set(profile['training_hashes']): errors.append('training_copy')
    for key,values in quantity_features(record,profile['kind']).items():
        bounds=profile.get('numeric_bounds',{}).get(key)
        if values and not bounds: errors.append('unseen_quantity_type:'+key)
        elif bounds and any(v<bounds['min'] or v>bounds['max'] for v in values): errors.append('out_of_training_range:'+key)
    return errors

def parse_output(raw):
    # No regex extraction/repair: the raw JSON-validity score must remain honest.
    def reject_constant(v): raise ValueError('Non-JSON constant '+v)
    try:
        value=json.loads(raw.strip(), parse_constant=reject_constant)
        return value if isinstance(value,dict) else None
    except (ValueError,TypeError): return None

def prompt(profile, attrs=None):
    spec='Return exactly one complete JSON object, without markdown or commentary. Create a new insurance dataset record. '
    spec+='Follow this JSON schema: '+canonical(profile['schema'])
    if attrs: spec+='\nRequired field values (dot paths): '+canonical(attrs)
    return [{'role':'system','content':spec},{'role':'user','content':'Generate synthetic data.'}]

def completion_tokens(tokenizer, messages, record):
    prefix=tokenizer.apply_chat_template(messages,tokenize=True,return_dict=False,add_generation_prompt=True,enable_thinking=False)
    full=tokenizer.apply_chat_template(messages+[{'role':'assistant','content':canonical(record)}],
                                       tokenize=True,return_dict=False,add_generation_prompt=False,enable_thinking=False)
    if full[:len(prefix)]!=prefix:
        raise ValueError('Chat template prefix mismatch; refusing to guess the completion loss mask.')
    if len(full)<=len(prefix): raise ValueError('Empty completion.')
    return {'input_ids':full,'attention_mask':[1]*len(full),'labels':[-100]*len(prefix)+full[len(prefix):]}

def choose_context(max_length):
    for length in (2048,4096,8192):
        if max_length<=length: return length
    raise ValueError(f'Complete records need {max_length} tokens (>8192); no truncation was applied.')

def select_candidate(candidates):
    if not candidates: raise ValueError('No completed evaluation candidates.')
    eligible=[c for c in candidates if c['raw_schema_validity']>=.95 and c.get('metric_records',1)>0]
    pool=eligible or candidates
    chosen=min(pool,key=lambda c:(c['normalized_score'],c['training_copy_rate'],c.get('validation_loss',math.inf)))
    return chosen, bool(eligible)


Writing /kaggle/working/insurance_gemma_program/src/core.py


In [3]:
%%writefile /kaggle/working/insurance_gemma_program/src/metrics.py
"""Distribution diagnostics; these are not factual/clinical correctness scores."""
import collections, hashlib, math, re, time
import numpy as np
from core import *

def tv(a,b):
    ca,cb=collections.Counter(a),collections.Counter(b)
    if not a or not b: return 1.0
    return .5*sum(abs(ca[k]/len(a)-cb[k]/len(b)) for k in ca.keys()|cb.keys())

def ks(a,b):
    if not a and not b: return 0.0
    if not a or not b: return 1.0
    grid=np.sort(np.concatenate([a,b]))
    return float(np.max(np.abs(np.searchsorted(np.sort(a),grid,side='right')/len(a)-
                               np.searchsorted(np.sort(b),grid,side='right')/len(b))))

def numeric_features(record,kind):
    return quantity_features(record,kind)

def vocabulary_distance(a,b,kind):
    def words(rs): return [w for r in rs for w in normalize(text(r,kind)).split()]
    return tv(words(a),words(b))

def style_features(records,kind):
    result=collections.defaultdict(list)
    tags=set(''.join(x['tags'] for x in records)) if kind=='bitext' else set()
    for r in records:
        t=text(r,kind); words=normalize(t).split(); grams=shingles(t)
        result['uppercase_fraction'].append(sum(c.isupper() for c in t)/max(sum(c.isalpha() for c in t),1))
        result['punctuation_per_word'].append(len(re.findall(r'[^\w\s]',t))/max(len(words),1))
        result['digit_fraction'].append(sum(c.isdigit() for c in t)/max(len(t),1))
        result['trigram_repetition'].append(1-len(grams)/max(len(words)-2,1))
        result['placeholder_count'].append(len(re.findall(r'\{\{[^{}]+\}\}',t)))
        if kind=='bitext':
            for tag in tags: result['tag_'+tag].append(int(tag in r['tags']))
    return dict(result)

class Embedder:
    """Embed all text using token chunks; do not silently discard long answers."""
    def __init__(self,config):
        from sentence_transformers import SentenceTransformer
        self.model=SentenceTransformer(config['embedding_id'],revision=config['embedding_revision'],device=config.get('embedding_device','cpu'))
        self.cache={}; self.chunk_size=min(200,self.model.max_seq_length-2)

    def encode(self,texts):
        pending=list(dict.fromkeys(t for t in texts if t not in self.cache))
        chunks=[]; owners=[]; weights=[]
        tokenizer=self.model.tokenizer
        for t in pending:
            ids=tokenizer.encode(t,add_special_tokens=False)
            for start in range(0,max(len(ids),1),self.chunk_size):
                part=ids[start:start+self.chunk_size]
                chunks.append(tokenizer.decode(part,skip_special_tokens=True) if part else ' ')
                owners.append(t); weights.append(max(len(part),1))
        if chunks:
            vec=self.model.encode(chunks,batch_size=64,normalize_embeddings=True,show_progress_bar=False,convert_to_numpy=True)
            sums={}; sizes=collections.Counter()
            for owner,w,v in zip(owners,weights,vec):
                sums[owner]=sums.get(owner,np.zeros_like(v))+w*v; sizes[owner]+=w
            for t,v in sums.items(): self.cache[t]=v/max(np.linalg.norm(v),1e-12)
        return np.stack([self.cache[t] for t in texts]) if texts else np.empty((0,self.model.get_sentence_embedding_dimension()))

def semantic_distance(a,b):
    # Biased MMD with a fixed RBF bandwidth on unit-normalized embeddings.
    # Natural finite-sample bias is handled by the real-real calibration.
    if not len(a) or not len(b): return 1.0
    def kernel(x,y): return np.exp(-(np.maximum(0,2-2*x@y.T))/2)
    return max(0.0,float(kernel(a,a).mean()+kernel(b,b).mean()-2*kernel(a,b).mean()))

class Evaluator:
    def __init__(self,profile,training,embedder,deadline=None):
        self.profile=profile; self.kind=profile['kind']; self.training=training; self.embedder=embedder
        self.hashes=set(profile['training_hashes'])
        self.calibrations={}
        self.deadline=deadline
        self.train_grams=[shingles(primary_text(r,self.kind)) for r in training]
        self.inverted=collections.defaultdict(set)
        for i,gs in enumerate(self.train_grams):
            for g in gs: self.inverted[g].add(i)

    def distances(self,a,b):
        if self.deadline is not None and time.time()>=self.deadline:
            raise TimeoutError('Metric budget exhausted; partial reports and adapters are retained.')
        paths=self.profile['condition_paths']
        categorical={p:tv([canonical(get_path(r,p)) for r in a],[canonical(get_path(r,p)) for r in b]) for p in paths}
        categorical['joint']=tv([canonical(condition(r,paths)) for r in a],[canonical(condition(r,paths)) for r in b])
        length={}
        for field in sorted(set(k for r in a+b for k in text_fields(r,self.kind))):
            length[field]=ks([len(normalize(text_fields(r,self.kind).get(field,'')).split()) for r in a],
                             [len(normalize(text_fields(r,self.kind).get(field,'')).split()) for r in b])
        fa=[numeric_features(r,self.kind) for r in a]; fb=[numeric_features(r,self.kind) for r in b]
        nums={}
        for key in sorted(set(k for fs in fa+fb for k,v in fs.items() if v)):
            av=[x for f in fa for x in f.get(key,[])]; bv=[x for f in fb for x in f.get(key,[])]
            nums[key]={'ks':ks(av,bv),'real_n':len(bv),'synthetic_n':len(av),
                'presence_gap':abs(sum(bool(f.get(key)) for f in fa)/max(len(fa),1)-sum(bool(f.get(key)) for f in fb)/max(len(fb),1)),
                'outside_reference_range':sum(x<min(bv) or x>max(bv) for x in av)/len(av) if av and bv else None}
        lm=list(length.values())+[v['ks'] for v in nums.values()]+[v['presence_gap'] for v in nums.values()]
        ea=self.embedder.encode([text(r,self.kind) for r in a]); eb=self.embedder.encode([text(r,self.kind) for r in b])
        components={'categorical':float(np.mean(list(categorical.values()))),
                    'length_numeric':float(np.mean(lm)) if lm else 0.,'semantic':semantic_distance(ea,eb)}
        return {'components':components,'categorical_tv':categorical,'length_ks':length,'numeric':nums}

    def calibrate(self,n,seed,repeats=30):
        cache_key=(n,seed,repeats)
        if cache_key in self.calibrations: return self.calibrations[cache_key]
        rng=random.Random(seed); values=[]
        for _ in range(repeats):
            if len(self.training)>=2*n:
                sample=rng.sample(self.training,2*n); a,b=sample[:n],sample[n:]
            else: a=rng.choices(self.training,k=n); b=rng.choices(self.training,k=n)
            values.append(self.distances(a,b)['components'])
        result={}
        for key in ('categorical','length_numeric','semantic'):
            xs=np.array([v[key] for v in values]); mean=float(xs.mean()); sd=float(xs.std())
            result[key]={'mean':mean,'std':sd,'q95':float(np.quantile(xs,.95)),'scale':max(mean+sd,1e-4)}
        self.calibrations[cache_key]={'sample_size':n,'repeats':repeats,'components':result,
                'method':'Equally sized disjoint training subsamples when possible; bootstrap otherwise.'}
        return self.calibrations[cache_key]

    def novelty(self,records):
        hashes=[fingerprint(r) for r in records]; near=[]
        for r in records:
            gs=shingles(primary_text(r,self.kind)); candidates=set()
            for g in gs: candidates.update(self.inverted.get(g,()))
            near.append(max((jaccard(gs,self.train_grams[i]) for i in candidates),default=0.))
        return {'training_copy_rate':sum(h in self.hashes for h in hashes)/max(len(hashes),1),
                'generated_duplicate_rate':1-len(set(hashes))/max(len(hashes),1) if hashes else 0.,
                'near_duplicate_question_rate':sum(x>=.85 for x in near)/max(len(near),1),
                'question_similarity_mean':float(np.mean(near)) if near else None}

    def consistency(self,records):
        if not records: return {'note':'No valid records.'}
        # Semantic relatedness is a diagnostic, never a correctness verdict.
        a=self.embedder.encode([text_fields(r,self.kind)['question'] for r in records])
        b=self.embedder.encode([text_fields(r,self.kind)['answer'] for r in records])
        metric='input_expected_output_cosine_mean' if self.kind=='reliability' else 'question_answer_cosine_mean'
        result={metric:float(np.sum(a*b,axis=1).mean()),
                'interpretation':'Relatedness only. Human review is required for factual and workflow correctness.'}
        if self.kind=='reliability':
            patterns={canonical((r['category'],r['expected_output']['routing_decision'],sorted(r['expected_output']['required_actions']))) for r in self.training}
            result['unseen_category_route_action_rate']=sum(canonical((r['category'],r['expected_output']['routing_decision'],sorted(r['expected_output']['required_actions']))) not in patterns for r in records)/len(records)
            result['source_policy_type_vs_insurance_line']={
                'generated':dict(collections.Counter(canonical((get_path(r,'input.context.policy_type'),get_path(r,'metadata.insurance_line'))) for r in records)),
                'training':dict(collections.Counter(canonical((get_path(r,'input.context.policy_type'),get_path(r,'metadata.insurance_line'))) for r in self.training))}
        return result

    def report(self,attempts,reference,calibration,seed=3407,bootstrap=30):
        schema_valid=[a for a in attempts if isinstance(a.get('record'),dict) and not schema_errors(a['record'],self.profile['schema'])]
        constraint_valid=[a for a in attempts if a.get('record') is not None and not validate_record(a['record'],self.profile,a.get('condition'))
                          and not a.get('length_limited') and not a.get('deadline_hit')]
        # Unknown labels/out-of-range values MUST remain in raw distribution metrics.
        valid=[a['record'] for a in schema_valid]
        # Metrics must not hide invalid JSON: validity and denominator are always reported.
        result={'attempts':len(attempts),'raw_schema_validity':len(schema_valid)/max(len(attempts),1),
                'raw_constraint_validity':len(constraint_valid)/max(len(attempts),1),'metric_records':len(valid),
                'constraint_errors':dict(collections.Counter(e for a in attempts for e in validate_record(a.get('record'),self.profile,a.get('condition')))),
                'reference_records':len(reference),'limitations':['Distribution metrics require structurally valid records; unknown labels and out-of-range values remain included. Consult raw validity alongside them.',
                'Numeric patterns cover explicit USD amounts, percentages, days, months, ages, and structured numeric fields; they do not infer all insurance quantities.']}
        if not valid:
            result.update(normalized_score=1e9,training_copy_rate=0.,status='no_valid_records'); return result
        # Compare equal sample counts, including after rejection; use matching calibration.
        matched=min(len(valid),len(reference));rng=random.Random(seed)
        sampled=rng.sample(valid,matched); reference=rng.sample(reference,matched)
        if calibration['sample_size']!=matched:
            calibration=self.calibrate(matched,seed,calibration['repeats'])
        result['matched_metric_sample_size']=matched
        result['calibration']=calibration
        distances=self.distances(sampled,reference); result.update(distances)
        score=float(np.mean([distances['components'][k]/calibration['components'][k]['scale'] for k in distances['components']]))
        result['normalized_score']=score; result.update(self.novelty(valid))
        result['within_real_real_95_envelope']={k:distances['components'][k]<=calibration['components'][k]['q95'] for k in distances['components']}
        result['consistency']=self.consistency(valid); result['vocabulary_tv']=vocabulary_distance(valid,reference,self.kind)
        sa,sb=style_features(valid,self.kind),style_features(reference,self.kind)
        result['style_ks']={k:ks(sa.get(k,[]),sb.get(k,[])) for k in sa.keys()|sb.keys()}
        # Both generated and reference samples are resampled. Small benchmark intervals remain descriptive.
        rng=random.Random(seed); scores=[]
        for _ in range(bootstrap):
            d=self.distances(rng.choices(sampled,k=matched),rng.choices(reference,k=matched))['components']
            scores.append(float(np.mean([d[k]/calibration['components'][k]['scale'] for k in d])))
        result['score_bootstrap_95_interval']=[float(x) for x in np.quantile(scores,[.025,.975])] if scores else None
        result['status']='measured';return result

def manual_review_sample(records,kind,n=50,seed=3407):
    by=collections.defaultdict(list)
    for record in records: by[record.get('domain') if kind=='insuranceqa' else record.get('intent') if kind=='bitext' else record.get('category')].append(record)
    rng=random.Random(seed)
    for group in by.values(): rng.shuffle(group)
    selected=[]
    while len(selected)<min(n,len(records)):
        for key in sorted(by,key=str):
            if by[key] and len(selected)<n: selected.append(by[key].pop())
    return [{'record':r,'schema_correct':'','on_topic':'','answer_or_routing_correct':'','plausible_new_example':'',
             'source_inconsistency_or_model_error':'','reviewer_notes':''} for r in selected]


Writing /kaggle/working/insurance_gemma_program/src/metrics.py


In [4]:
%%writefile /kaggle/working/insurance_gemma_program/src/model_runtime.py
"""GPU imports are isolated from notebook orchestration and CPU data checks."""
import os, time
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('UNSLOTH_COMPILE_DISABLE','1')
os.environ.setdefault('WANDB_DISABLED','true')
from core import *

def load_model(config, adapter=None, train=False):
    # Unsloth must patch Transformers before the latter is imported.
    from unsloth import FastModel
    import torch
    from peft import PeftModel
    rank=int(os.environ.get('LOCAL_RANK','0'))
    torch.cuda.set_device(rank)
    # An exact local snapshot prevents automatic repo/quantized-checkpoint substitution.
    model_path=config.get('model_path')
    if not model_path or not Path(model_path).is_dir():
        from huggingface_hub import snapshot_download
        model_path=snapshot_download(config['model_id'],revision=config['model_revision'],
            allow_patterns=['*.json','*.safetensors','*.model','*.jinja','*.txt'])
    model,processor=FastModel.from_pretrained(
        model_name=model_path,
        max_seq_length=config.get('context_length',8192),dtype=torch.float16,
        load_in_4bit=True,full_finetuning=False,
        device_map={'':rank},attn_implementation='sdpa',
        use_gradient_checkpointing='unsloth' if train else False,
    )
    tokenizer=getattr(processor,'tokenizer',processor)
    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
    tokenizer.padding_side='right'
    if adapter:
        model=PeftModel.from_pretrained(model,str(adapter),is_trainable=train)
    elif train:
        model=FastModel.get_peft_model(model,finetune_vision_layers=False,finetune_language_layers=True,
            finetune_attention_modules=True,finetune_mlp_modules=True,
            r=config['rank'],lora_alpha=config['alpha'],lora_dropout=.05,
            bias='none',random_state=config['seed'])
    if train:
        bad=[n for n,p in model.named_parameters() if p.requires_grad and
             ('lora_' not in n or any(s in n for s in ('vision_tower','audio_tower','embed_vision','embed_audio')))]
        if bad: raise RuntimeError('Unexpected trainable parameters: '+str(bad[:10]))
        if not any(p.requires_grad for p in model.parameters()): raise RuntimeError('No trainable adapter parameters.')
        model.config.use_cache=False
    else:
        model.eval(); FastModel.for_inference(model)
    return model,tokenizer

class Generator:
    def __init__(self, config, profile, adapter=None, temperature=.9):
        self.config=config; self.profile=profile; self.temperature=temperature
        self.model,self.tokenizer=load_model(config,adapter=adapter)
        self.train_hashes=set(profile['training_hashes'])

    def one(self, attrs, seed, greedy=False, max_tokens=None, deadline=None):
        import torch
        from transformers import set_seed, StoppingCriteria, StoppingCriteriaList
        class Deadline(StoppingCriteria):
            def __call__(self,input_ids,scores,**kwargs):
                return deadline is not None and time.time()>=deadline
        set_seed(seed)
        ids=self.tokenizer.apply_chat_template(prompt(self.profile,attrs),tokenize=True,return_dict=False,
                add_generation_prompt=True,enable_thinking=False)
        remaining=self.config['context_length']-len(ids)
        limit=min(max_tokens or self.config['max_new_tokens'],remaining)
        if limit<1: raise ValueError('Prompt consumes the configured context window.')
        inputs=torch.tensor([ids],device=self.model.device)
        kwargs={'max_new_tokens':limit,'do_sample':not greedy,'repetition_penalty':1.0,
                'pad_token_id':self.tokenizer.pad_token_id,'use_cache':True,
                'stopping_criteria':StoppingCriteriaList([Deadline()])}
        if not greedy: kwargs.update(temperature=self.temperature,top_p=.95,top_k=0)
        with torch.inference_mode():
            outputs=self.model.generate(input_ids=inputs,attention_mask=torch.ones_like(inputs),**kwargs)
        tokens=outputs[0,len(ids):].tolist()
        raw=self.tokenizer.decode(tokens,skip_special_tokens=True).strip()
        return {'raw':raw,'tokens':len(tokens),'length_limited':len(tokens)>=limit,
                'deadline_hit':deadline is not None and time.time()>=deadline,'record':parse_output(raw)}

    def generate_synthetic(self,n,seed,mode='validated',conditions=None,deadline=None,trace_path=None):
        if mode not in ('raw','validated'): raise ValueError('mode must be raw or validated')
        if n<0: raise ValueError('n must be nonnegative')
        attrs=conditions if conditions is not None else sample_conditions(self.profile,n,seed)
        if len(attrs)!=n: raise ValueError('Condition count does not match n')
        records=[]; attempts=[]; completed=0
        for i,cond in enumerate(attrs):
            if deadline and time.time()>=deadline: break
            for retry in range(1 if mode=='raw' else 6):  # initial attempt plus five retries
                if deadline and time.time()>=deadline: break
                result=self.one(cond,seed+i*1009+retry,deadline=deadline)
                errors=validate_record(result['record'],self.profile,cond,reject_copy=(mode=='validated'))
                if result['length_limited']: errors.append('length_limited')
                if result['deadline_hit']: errors.append('deadline_hit')
                attempt={**result,'request_index':i,'retry':retry,'condition':cond,'errors':errors,'accepted':not errors}
                attempts.append(attempt)
                if trace_path:
                    with Path(trace_path).open('a',encoding='utf-8') as f: f.write(canonical(attempt)+'\n')
                if mode=='raw' or not errors:
                    # Raw outputs remain available even when the record is invalid JSON.
                    if result['record'] is not None and (mode=='raw' or not errors): records.append(result['record'])
                    break
            completed+=1
        self.last_report={'requested':n,'requests_processed':completed,'attempts':len(attempts),
            'accepted':sum(a['accepted'] for a in attempts),'returned':len(records),
            'shortfall':n-len(records),'rejections':dict(collections.Counter(e for a in attempts for e in a['errors'])),
            'complete':completed==n,'mode':mode}
        self.last_attempts=attempts
        return records

def exported_records(records, kind, seed):
    if kind!='reliability': return records
    return [dict(r,id=f'synthetic-{seed}-{i:06d}') for i,r in enumerate(records)]


Writing /kaggle/working/insurance_gemma_program/src/model_runtime.py


In [5]:
%%writefile /kaggle/working/insurance_gemma_program/src/prepare.py
"""Download pinned public sources, prepare records, and audit splits."""
import argparse, collections, csv, gzip, hashlib, json, shutil, time, urllib.request
from pathlib import Path
from core import *

def fetch(url, path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists(): return path
    last=None
    for attempt in range(3):
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'insurance-gemma-research/1.0'})
            with urllib.request.urlopen(req,timeout=90) as source, path.with_suffix(path.suffix+'.part').open('wb') as target:
                shutil.copyfileobj(source,target)
            path.with_suffix(path.suffix+'.part').replace(path); return path
        except Exception as exc: last=exc; time.sleep(2**attempt)
    raise RuntimeError(f'Download failed: {url}') from last

def decoded(value, vocabulary):
    words=value.split()
    missing=[w for w in words if w.startswith('idx_') and w not in vocabulary]
    if missing: raise ValueError(f'Unknown vocabulary IDs: {missing[:5]}')
    return ' '.join(vocabulary.get(w,w) for w in words)

def load_qa(raw, revision):
    base=f'https://raw.githubusercontent.com/shuzi/insuranceQA/{revision}/'
    vocab_path=fetch(base+'V2/vocabulary',raw/'V2/vocabulary')
    with vocab_path.open(encoding='utf-8') as f:
        vocabulary=dict(line.rstrip('\n\r').split('\t',1) for line in f if line.strip())
    answer_path=fetch(base+'V2/InsuranceQA.label2answer.raw.encoded.gz',raw/'V2/InsuranceQA.label2answer.raw.encoded.gz')
    answers={}
    with gzip.open(answer_path,'rt',encoding='utf-8') as f:
        for line in f:
            label,encoded=line.rstrip('\n\r').split('\t',1)
            answers[label]=decoded(encoded,vocabulary)
    rows=[]
    for source_split,split in [('train','train'),('valid','validation'),('test','test')]:
        name=f'V2/InsuranceQA.question.anslabel.raw.100.pool.solr.{source_split}.encoded.gz'
        path=fetch(base+name,raw/name)
        with gzip.open(path,'rt',encoding='utf-8') as f:
            for line_no,line in enumerate(f,1):
                parts=line.rstrip('\r\n').split('\t')
                if len(parts)!=4: raise ValueError(f'{name}:{line_no}: expected four tab-separated fields')
                domain,question,truth,_distractors=parts
                labels=list(dict.fromkeys(truth.split()))
                if not labels or any(k not in answers for k in labels): raise ValueError(f'Unresolvable ground truth: {name}:{line_no}')
                rows.append({'uid':f'{source_split}:{line_no}','split':split,
                             'record':{'domain':domain,'question':decoded(question,vocabulary),'answers':[answers[k] for k in labels]}})
    return rows

def load_bitext(raw, revision, seed):
    name='bitext-insurance-llm-chatbot-training-dataset.csv'
    path=fetch(f'https://huggingface.co/datasets/bitext/Bitext-insurance-llm-chatbot-training-dataset/resolve/{revision}/{name}',raw/name)
    keys=['instruction','intent','category','tags','response']
    with path.open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        if not set(keys)<=set(reader.fieldnames or []): raise ValueError(f'Unexpected Bitext columns: {reader.fieldnames}')
        records=[{k:r[k] for k in keys} for r in reader]
    if any(not r['instruction'].strip() or not r['response'].strip() for r in records): raise ValueError('Empty Bitext text records.')
    assignments,groups=grouped_split(records,'bitext',seed)
    rows=[{'uid':f'csv:{i+2}','split':assignments[i],'record':r} for i,r in enumerate(records)]
    manifest=[{'uid':x['uid'],'source_split':'train','split':x['split'],'group':groups[i],
               'sha256':fingerprint(x['record']),'reason':None} for i,x in enumerate(rows)]
    return rows,manifest

def load_reliability(raw, revision):
    rows=[]
    for split in ('train','validation','test'):
        name=f'data/{split}.jsonl'
        path=fetch(f'https://huggingface.co/datasets/pashas/insurance-ai-reliability-benchmark/resolve/{revision}/{name}',raw/name)
        for i,record in enumerate(read_jsonl(path)):
            if not {'id','category','subcategory','difficulty','input','expected_output','evaluation_criteria','metadata'}<=record.keys():
                raise ValueError(f'Unexpected benchmark schema at {split}:{i}')
            uid=f'{split}:{record.pop("id")}'
            rows.append({'uid':uid,'split':split,'record':record})
    return rows

def summarize(rows, kind):
    paths={'insuranceqa':['domain'],'bitext':['intent','category'],'reliability':['category','difficulty','expected_output.routing_decision']}[kind]
    result={}
    for split in ('train','validation','test'):
        records=[r['record'] for r in rows if r['split']==split]
        result[split]={'n':len(records),'categories':{p:dict(collections.Counter(str(get_path(r,p)) for r in records)) for p in paths},
                       'primary_word_lengths':sorted(len(normalize(primary_text(r,kind)).split()) for r in records)}
    return result

def prepare(config):
    root=Path(config['run_dir']); raw=root/'raw'; raw.mkdir(parents=True,exist_ok=True)
    kind=config['dataset']; revision=config['dataset_revision']
    if kind=='bitext': rows,manifest=load_bitext(raw,revision,config['seed']); before=summarize(rows,kind)
    else:
        rows=load_qa(raw,revision) if kind=='insuranceqa' else load_reliability(raw,revision)
        before=summarize(rows,kind); rows,manifest=audit_official(rows,kind)
    records={s:[x['record'] for x in rows if x['split']==s] for s in ('train','validation','test')}
    if any(not x for x in records.values()): raise ValueError('A split is empty after grouping; inspect split_manifest.jsonl.')
    profile=fit_profile(records['train'],kind)
    for s,rs in records.items(): write_jsonl(root/'prepared'/f'{s}.jsonl',rs)
    write_json(root/'profile.json',profile); write_jsonl(root/'split_manifest.jsonl',manifest)
    sources=[{'path':str(p.relative_to(raw)),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
             for p in sorted(raw.rglob('*')) if p.is_file()]
    unseen={s:sum(canonical(condition(r,profile['condition_paths'])) not in profile['joint_counts'] for r in rs) for s,rs in records.items()}
    report={'dataset':kind,'revision':revision,'source_files':sources,'before':before,'after':summarize(rows,kind),
            'excluded':sum(m['split']=='excluded' for m in manifest),'unseen_condition_tuples':unseen,
            'target':'Retained training records with their empirical row frequencies.',
            'grouping':'Exact normalized word-trigram Jaccard >=0.85; connected components.',
            'notes':['Source inconsistencies are preserved and must be distinguished from model errors.',
                     'No resampling to equal class frequencies. No generated training examples.']}
    if kind=='insuranceqa':
        train_answers={normalize(a) for r in records['train'] for a in r['answers']}
        report['shared_answer_records']={s:sum(any(normalize(a) in train_answers for a in r['answers']) for r in records[s]) for s in ('validation','test')}
        report['notes'].append('Questions are leakage-controlled; shared professional answers are separately counted, not silently removed.')
    write_json(root/'preparation_report.json',report)
    print(json.dumps({'dataset':kind,'counts':{s:len(v) for s,v in records.items()},'excluded':report['excluded']},indent=2),flush=True)
    return report

if __name__=='__main__':
    parser=argparse.ArgumentParser(); parser.add_argument('--config',required=True)
    args=parser.parse_args(); prepare(read_json(args.config))


Writing /kaggle/working/insurance_gemma_program/src/prepare.py


In [6]:
%%writefile /kaggle/working/insurance_gemma_program/src/run.py
"""Single-session controller. Child processes own GPU contexts."""
import argparse, concurrent.futures, csv, importlib.metadata, json, os, platform, shutil, signal, subprocess, sys, time, traceback
from pathlib import Path
from core import *
HERE=Path(__file__).resolve().parent

def command(args, log, deadline, env=None):
    log=Path(log); log.parent.mkdir(parents=True,exist_ok=True)
    timeout=deadline-time.time()
    if timeout<=0: raise TimeoutError('Session work budget exhausted before '+str(args))
    with log.open('a',encoding='utf-8') as handle:
        handle.write('\nCOMMAND '+repr(args)+'\n');handle.flush()
        # Inherit the controller's process group so interrupting the notebook kills descendants too.
        proc=subprocess.Popen(args,stdout=handle,stderr=subprocess.STDOUT,env=env)
        try:
            code=proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            proc.terminate()  # torchrun forwards SIGTERM to its distributed workers.
            try: proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            raise TimeoutError(f'Budget exhausted; see {log}')
    if code: raise RuntimeError(f'Subprocess exited {code}; see {log}\n'+log.read_text(encoding='utf-8',errors='replace')[-6000:])

def preflight(config,root):
    # nvidia-smi instead of torch: importing/initialising CUDA here would keep a context alive on both
    # GPUs for the whole run and take VRAM away from the training processes.
    rows=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader,nounits'],text=True).strip().splitlines()
    if len(rows)!=2: raise RuntimeError('Select GPU T4 x2 in Kaggle Settings. Exactly two visible GPUs are required.')
    devices=[]
    for row in rows:
        name,mem,driver=[x.strip() for x in row.split(',')]
        devices.append({'name':name,'memory_gib':float(mem)/1024,'driver':driver})
        if float(mem)<14*1024: raise RuntimeError('Each GPU must have approximately 16 GB VRAM.')
    report={'python':sys.version,'platform':platform.platform(),'torch':importlib.metadata.version('torch'),'devices':devices,
            'packages':{d.metadata['Name']:d.version for d in importlib.metadata.distributions() if d.metadata.get('Name')}}
    write_json(root/'environment.json',report)
    freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
    (root/'environment.lock.txt').write_text(freeze,encoding='utf-8')

class Terminated(BaseException):
    """SIGTERM from the notebook watchdog or a session stop. BaseException so stage-level handlers cannot swallow it."""

def _term(signum,frame):
    raise Terminated('Controller received SIGTERM; preserving what exists.')

def read_jsonl_tolerant(path):
    rows=[]
    with Path(path).open(encoding='utf-8-sig') as handle:
        for line in handle:
            if not line.strip(): continue
            try: rows.append(json.loads(line))
            except ValueError: break   # trailing line cut off by a kill/timeout
    return rows

def safe_parallel(config_path,config,groups,deadline,errors,label):
    """A generation-stage failure must not discard a trained adapter: record it and carry on with what finished."""
    try: parallel_jobs(config_path,config,groups,deadline)
    except Exception as exc:
        errors.append({'stage':label,'error':str(exc)[-3000:]})
        print(f'[{time.strftime("%H:%M:%S")}] {label}: stopped early ({type(exc).__name__}); continuing with whatever finished.',flush=True)

def parallel_jobs(config_path,config,groups,deadline):
    root=Path(config['run_dir'])
    def slot(gpu,items):
        for name,job in items:
            if time.time()>=deadline: return
            path=root/'jobs'/f'{name}.json';write_json(path,job)
            env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']=str(gpu)
            env.pop('LOCAL_RANK',None);env.pop('RANK',None);env.pop('WORLD_SIZE',None)
            command([sys.executable,str(HERE/'worker.py'),'--config',str(config_path),'--job',str(path)],
                    root/'logs'/f'{name}.log',deadline,env)
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
        futures=[pool.submit(slot,gpu,groups[gpu::2]) for gpu in range(2)]
        for f in futures: f.result()

def export(config,chosen,passed,root):
    destination=root/'export'; adapter=destination/'adapter';destination.mkdir(exist_ok=True)
    source=Path(chosen['adapter'])
    if adapter.exists(): raise RuntimeError('Export adapter already exists; use a new run directory.')
    adapter.mkdir()
    for name in ('adapter_config.json','adapter_model.safetensors'):
        if not (source/name).exists(): raise RuntimeError(f'Missing adapter file: {source/name}')
        shutil.copy2(source/name,adapter/name)
    # Avoid a machine-specific base path in the exported PEFT configuration.
    ac=read_json(adapter/'adapter_config.json');ac['base_model_name_or_path']=config['model_id'];ac['revision']=config['model_revision']
    write_json(adapter/'adapter_config.json',ac)
    shutil.copytree(root/'tokenizer',destination/'tokenizer',dirs_exist_ok=True)
    settings={**config,'temperature':chosen['temperature'],'adapter_path':'adapter',
              'validation_acceptance_passed':passed,'selection':chosen}
    settings.pop('model_path',None);settings.pop('training_deadline',None);settings.pop('started_at',None)
    write_json(destination/'generation_config.json',settings)
    for name in ('profile.json','environment.lock.txt','environment.json','preparation_report.json','length_report.json'):
        if (root/name).exists(): shutil.copy2(root/name,destination/name)
    for p in HERE.glob('*.py'): shutil.copy2(p,destination/p.name)
    (destination/'load_generator.py').write_text('''from pathlib import Path
from core import read_json
from model_runtime import Generator, exported_records
ROOT = Path(__file__).resolve().parent
config = read_json(ROOT / "generation_config.json")
profile = read_json(ROOT / "profile.json")
generator = Generator(config, profile, adapter=ROOT / "adapter", temperature=config["temperature"])
def generate_synthetic(n, seed, mode="validated"):
    records = generator.generate_synthetic(n, seed, mode)
    return exported_records(records, profile["kind"], seed)
''',encoding='utf-8')
    return destination

def flat_csv(path,records):
    if not records: return
    if any(isinstance(v,(dict,list)) for r in records for v in r.values()): return
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=sorted(set().union(*(r.keys() for r in records))));writer.writeheader();writer.writerows(records)

def pipeline(config_path,prepare_only=False):
    config=read_json(config_path);root=Path(config['run_dir']);root.mkdir(parents=True,exist_ok=True)
    resume=bool(config.get('resume'))
    if (root/'session_status.json').exists() and not resume:
        raise RuntimeError('Run directory already used. Choose a new RUN_NAME; existing results are preserved.')
    signal.signal(signal.SIGTERM,_term)
    started=time.time();total=config['hours']*3600;deadline=started+total
    # Budget map (fractions of the work budget): training <= 60%, validation selection <= 80%,
    # final generation <= 96%; the rest is CPU metrics, packaging and slack.  Every stage ends
    # early when its work is done; these are ceilings, and every later stage degrades gracefully.
    # train_until_done: training owns the session (all epochs) except a 25-minute reserve; later stages get what is left.
    train_end=deadline-1500 if config.get('train_until_done') else started+total*config.get('train_fraction',.60)
    selection_end=started+total*config.get('selection_fraction',.80)
    final_end=started+total*config.get('final_fraction',.96)
    config.update(started_at=started,training_deadline=train_end)
    write_json(config_path,config)
    if resume:
        # Keep evidence from the interrupted session; stale stage outputs must not pass as new ones.
        for name in ('failure_report.json','session_status.json'):
            if (root/name).exists(): shutil.copy2(root/name,root/(name+'.previous'))
        (root/'failure_report.json').unlink(missing_ok=True)
        if (root/'export').exists(): shutil.move(str(root/'export'),str(root/f'export_superseded_{int(started)}'))
    write_json(root/'session_status.json',{'status':'running','stage':'preparation','started_at':started,'resume':resume})
    status={'status':'running'};stage='preparation'
    def mark(name):
        nonlocal stage
        stage=name;print(f'[{time.strftime("%H:%M:%S")}] {name}',flush=True)
        write_json(root/'session_status.json',{'status':'running','stage':stage,'elapsed_hours':(time.time()-started)/3600,'resume':resume})
    try:
        from prepare import prepare
        mark('preparation')
        have_data=(root/'profile.json').exists() and all((root/'prepared'/f'{s}.jsonl').exists() for s in ('train','validation','test'))
        if resume and have_data: print('Resume: reusing prepared data and profile.',flush=True)
        else: prepare(config)
        if prepare_only:
            status={'status':'preparation_complete','gpu_training':'not_run'};return
        mark('hardware_and_environment');preflight(config,root)
        mark('tokenization_and_context_check')
        from train import tokenize_data
        have_tokens=(all((root/'tokens'/f'{s}.jsonl').exists() for s in ('train','validation')) and (root/'tokenizer').is_dir()
                     and (root/'length_report.json').exists() and config.get('context_length') and config.get('max_new_tokens'))
        if resume and have_tokens: print('Resume: reusing token files.',flush=True)
        else: config=tokenize_data(config);write_json(config_path,config)
        # Pin and cache the exact original model before Unsloth can choose a substitute repo.
        mark('download_exact_base_weights')
        from huggingface_hub import snapshot_download
        config['model_path']=snapshot_download(config['model_id'],revision=config['model_revision'],
            allow_patterns=['*.json','*.safetensors','*.model','*.jinja','*.txt'])
        write_json(config_path,config)
        train_command=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',str(HERE/'train.py'),'--config',str(config_path)]
        if resume: print('Resume: skipping smoke tests (already passed in the interrupted session).',flush=True)
        else:
            mark('two_gpu_smoke_training')
            command(train_command+['--smoke'],root/'logs/smoke.log',deadline-60)
            mark('fresh_process_smoke_reload')
            parallel_jobs(config_path,config,[('smoke_reload',{'adapter':str(root/'smoke_adapter'),'verify':True,
                'output':str(root/'smoke_reload.json'),'deadline':deadline-60})],deadline-60)
        def training_finished():
            rp=root/'training_report.json';sp=root/'checkpoints/shortlist.json'
            if not (rp.exists() and sp.exists()): return False
            if read_json(rp).get('stop_reason') not in ('epoch_limit','early_stopping'): return False
            return all(Path(x['path']).exists() for x in read_json(sp))
        if resume and training_finished():
            print('Resume: training already finished all epochs; going straight to evaluation.',flush=True)
        else:
            mark('training')
            command(train_command+(['--resume'] if resume else []),root/'logs/train.log',deadline-60)
        if (root/'final_adapter/adapter_model.safetensors').exists():
            shutil.make_archive(str(root/'final_training_adapter'),'zip',root/'final_adapter')
        if config.get('train_until_done'):
            left=max(deadline-time.time()-1200,0)   # split whatever time remains after training
            selection_end=time.time()+left*.45;final_end=time.time()+left*.95
        shortlist=read_json(root/'checkpoints/shortlist.json')
        if not shortlist: raise RuntimeError('Training produced no evaluated checkpoints.')
        stage_errors=[]
        profile=read_json(root/'profile.json');validation=read_jsonl(root/'prepared/validation.jsonl')
        n=min(config['selection_n'],len(validation));seed=config['seed']+10
        attrs=sample_conditions(profile,n,seed);reference=random.Random(seed).sample(validation,n)
        # Adapters first: the untuned base is only a control and is the first thing to give up if time runs short.
        models=[{'name':f'adapter_{i}','path':x['path'],'validation_loss':x['validation_loss']} for i,x in enumerate(shortlist)]
        if config.get('evaluate_base',True): models.append({'name':'base','path':None,'validation_loss':None})
        temperatures=config.get('selection_temperatures',[.9,.7,1.])
        selection_deadline=min(max(selection_end,time.time()+900),deadline-1800)
        descriptors=[];groups=[]
        for model in models:
            generations=[]
            for temperature in temperatures:
                label=f'{model["name"]}_t{temperature:.1f}';out=root/'selection'/f'{label}.jsonl'
                generations.append({'output':str(out),'conditions':attrs,'seed':seed,'temperature':temperature})
                descriptors.append({'name':label,'adapter':model['path'],'validation_loss':model['validation_loss'],
                                    'temperature':temperature,'path':str(out)})
            groups.append((model['name'],{'adapter':model['path'],'generations':generations,'deadline':selection_deadline}))
        mark('validation_generation');safe_parallel(config_path,config,groups,selection_deadline+300,stage_errors,'validation_generation')
        mark('validation_distribution_metrics')
        from metrics import Embedder,Evaluator,manual_review_sample
        training=read_jsonl(root/'prepared/train.jsonl')
        evaluator=Evaluator(profile,training,Embedder(config),deadline=deadline-120);calibration=evaluator.calibrate(n,seed,config['bootstrap_repeats'])
        write_json(root/'calibration_validation.json',calibration)
        candidates=[];baseline=[];incomplete=[];partial=[]
        min_attempts=int(config.get('min_partial_attempts',16))
        for item in descriptors:
            path=Path(item['path']);status_path=Path(str(path)+'.status.json')
            complete=status_path.exists() and bool(read_json(status_path).get('complete'))
            attempts=[a for a in (read_jsonl_tolerant(path) if path.exists() else []) if not a.get('deadline_hit')]
            if not complete and len(attempts)<min_attempts:
                incomplete.append(item['name']);continue
            if not complete: partial.append(item['name'])
            report=evaluator.report(attempts,reference,calibration,seed,config['bootstrap_repeats'])
            report.update(item,partial=not complete);write_json(path.with_suffix('.metrics.json'),report)
            (candidates if item['adapter'] else baseline).append(report)
        if candidates: chosen,passed=select_candidate(candidates)
        else:
            best=shortlist[0]
            chosen={'name':'best_validation_loss_fallback','adapter':best['path'],'validation_loss':best['validation_loss'],'temperature':.9,
                    'raw_schema_validity':None,'normalized_score':None,'training_copy_rate':None,'diagnostic_only':True,
                    'reason':'No validation generation finished inside its time budget; the lowest-validation-loss checkpoint is exported unranked.'}
            passed=False
        selection={'chosen':chosen,'validation_acceptance_passed':passed,'incomplete_candidates':incomplete,'partial_candidates':partial,
                   'baseline':baseline,'candidates':candidates,'calibration':calibration,'stage_errors':stage_errors,
                   'selection_complete':not incomplete and not partial and len(shortlist)==2,
                   'note':'Only adapters are export candidates; the untuned base is a control. Partial candidates were scored on fewer samples.'}
        write_json(root/'selection_report.json',selection)
        mark('export_and_fresh_process_reload');destination=export(config,chosen,passed,root)
        # Secure a downloadable adapter now; the final package below replaces it once reports exist.
        shutil.make_archive(str(root/'adapter_package_preliminary'),'zip',destination)
        reload_job={'adapter':str(destination/'adapter'),'verify':True,'output':str(root/'export_reload.json'),'deadline':deadline-120}
        safe_parallel(config_path,config,[('export_reload',reload_job)],deadline-60,stage_errors,'export_reload')
        # Test content is first opened for scoring only after all model/sampler choices are locked.
        test=read_jsonl(root/'prepared/test.jsonl');ntest=min(config['final_n'],len(test))
        attrs=sample_conditions(profile,ntest,config['seed']+100)
        raw_path=root/'final/raw.jsonl';accepted_path=root/'final/validated_attempts.jsonl'
        final_deadline=min(max(final_end,time.time()+900),deadline-900)
        def final_job(path,mode):
            return {'adapter':str(destination/'adapter'),'deadline':final_deadline,'generations':[
                {'output':str(path),'conditions':attrs,'seed':config['seed']+100,'temperature':chosen['temperature'],'mode':mode}]}
        # Raw and validated generation run in parallel, one per GPU (previously one GPU did both while the other idled).
        mark('final_raw_and_validated_generation')
        safe_parallel(config_path,config,[('final_raw',final_job(raw_path,'raw')),('final_validated',final_job(accepted_path,'validated'))],
                      final_deadline+300,stage_errors,'final_generation')
        mark('final_test_metrics')
        reference=random.Random(config['seed']+100).sample(test,ntest)
        calibration=evaluator.calibrate(ntest,config['seed']+100,config['bootstrap_repeats'])
        final_report={'test_size':ntest,'validation_acceptance_passed':passed,'selection_complete':selection['selection_complete'],
                      'human_review_complete':False,'all_distribution_requirements_proven':False,'stage_errors':stage_errors}
        raw_attempts=[a for a in (read_jsonl_tolerant(raw_path) if raw_path.exists() else []) if not a.get('deadline_hit')]
        accepted_attempts=[a for a in (read_jsonl_tolerant(accepted_path) if accepted_path.exists() else []) if not a.get('deadline_hit')]
        accepted=[a for a in accepted_attempts if a['accepted']]
        final_report['raw']=evaluator.report(raw_attempts,reference,calibration,config['seed'],config['bootstrap_repeats'])
        final_report['accepted']=evaluator.report(accepted,reference,calibration,config['seed'],config['bootstrap_repeats'])
        final_report['generation_status']={name:read_json(str(path)+'.status.json') if Path(str(path)+'.status.json').exists() else {'complete':False}
            for name,path in [('raw',raw_path),('validated',accepted_path)]}
        final_report['complete']=all(s['complete'] for s in final_report['generation_status'].values())
        final_report['rejection_rate']=1-len(accepted)/max(len(accepted_attempts),1)
        final_report['accepted_schema_checks_passed']=all(not validate_record(a['record'],profile,a['condition'],reject_copy=True) for a in accepted)
        from model_runtime import exported_records
        records=exported_records([a['record'] for a in accepted],profile['kind'],config['seed']+100)
        (root/'final').mkdir(parents=True,exist_ok=True)
        write_jsonl(root/'final/synthetic.jsonl',records);flat_csv(root/'final/synthetic.csv',records)
        review=manual_review_sample(records,profile['kind']);write_jsonl(root/'final/manual_review.jsonl',review)
        final_report['manual_review_records']=len(review)
        write_json(root/'final_report.json',final_report)
        for name in ('selection_report.json','final_report.json','training_report.json','export_reload.json'):
            if (root/name).exists(): shutil.copy2(root/name,destination/name)
        shutil.copytree(root/'final',destination/'samples_and_review',dirs_exist_ok=True)
        (destination/'README.md').write_text('# Insurance synthetic-data adapter\n\n'
            'Install the saved environment in a compatible Linux/CUDA runtime. Run `from load_generator import generate_synthetic, generator`.\n'
            'Use `generate_synthetic(10, seed=42)` and inspect `generator.last_report` for shortfalls.\n'
            'This package includes an adapter, not the base weights. Base model ID and revision are pinned in generation_config.json.\n'
            'Read final_report.json before using outputs. Automated metrics do not prove correctness or privacy.\n',encoding='utf-8')
        shutil.make_archive(str(root/'adapter_package'),'zip',destination)
        status={'status':'completed' if final_report['complete'] and selection['selection_complete'] and not stage_errors else 'partial',
                'validation_acceptance_passed':passed,'training_stop_reason':read_json(root/'training_report.json')['stop_reason'],
                'package':str(root/'adapter_package.zip'),'stage_errors':stage_errors,'quality_status':'Human review and interpretation required.'}
    except (Exception,Terminated) as exc:
        status={'status':'budget_exhausted' if isinstance(exc,(TimeoutError,Terminated)) else 'failed','stage':stage,
                'error':str(exc),'traceback':traceback.format_exc()}
        write_json(root/'failure_report.json',status)
        # Preserve a usable diagnostic adapter even when generation/evaluation runs out of time.
        try:
            recovery=root/'export'
            if not recovery.exists() and (root/'checkpoints/shortlist.json').exists():
                best=read_json(root/'checkpoints/shortlist.json')[0]
                recovery=export(config,{'adapter':best['path'],'temperature':.9,'validation_loss':best['validation_loss'],
                    'diagnostic_only':True,'reason':'Generation selection did not finish.'},False,root)
            elif not recovery.exists():
                # A kernel/session interruption can occur before the first evaluation. Preserve the latest periodic checkpoint anyway.
                checkpoints=sorted((root/'checkpoints').glob('checkpoint-*'),key=lambda p:int(p.name.split('-')[-1])) if (root/'checkpoints').exists() else []
                if checkpoints:
                    recovery=export(config,{'adapter':str(checkpoints[-1]),'temperature':.9,'validation_loss':None,
                        'diagnostic_only':True,'reason':'Recovered latest periodic checkpoint before ranking.'},False,root)
            if recovery.exists():
                shutil.copy2(root/'failure_report.json',recovery/'failure_report.json')
                (recovery/'README_DIAGNOSTIC_ONLY.md').write_text('This run did not complete evaluation. Read failure_report.json. '
                    'The adapter is preserved for diagnosis or a later evaluation, not certified synthetic-data generation.',encoding='utf-8')
                archive=shutil.make_archive(str(root/'adapter_DIAGNOSTIC_ONLY'),'zip',recovery)
                status['diagnostic_adapter_package']=archive
        except Exception as recovery_error: status['recovery_error']=str(recovery_error)
        raise
    finally:
        status['elapsed_hours']=(time.time()-started)/3600
        write_json(root/'session_status.json',status)
        print(json.dumps(status,indent=2),flush=True)

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--config',required=True);p.add_argument('--prepare-only',action='store_true')
    args=p.parse_args();pipeline(Path(args.config).resolve(),args.prepare_only)


Writing /kaggle/working/insurance_gemma_program/src/run.py


In [7]:
%%writefile /kaggle/working/insurance_gemma_program/src/train.py
"""Two-process QLoRA SFT, budget control, top-two checkpoints, and smoke checks."""
import argparse, hashlib, math, os, shutil, time
from pathlib import Path
os.environ.setdefault('UNSLOTH_COMPILE_DISABLE','1')
from core import *

def tokenize_data(config):
    from transformers import AutoTokenizer
    root=Path(config['run_dir']); profile=read_json(root/'profile.json')
    tok=AutoTokenizer.from_pretrained(config['model_id'],revision=config['model_revision'])
    maxima={}; completion_max=0; prompt_max=0
    for split in ('train','validation'):
        encoded=[]; lengths=[]
        for i,record in enumerate(read_jsonl(root/'prepared'/f'{split}.jsonl')):
            variants=[completion_tokens(tok,prompt(profile,None),record),
                      completion_tokens(tok,prompt(profile,condition(record,profile['condition_paths'])),record)]
            lengths.extend(len(x['input_ids']) for x in variants)
            for x in variants:
                supervised=sum(y!=-100 for y in x['labels']); completion_max=max(completion_max,supervised)
                prompt_max=max(prompt_max,len(x['input_ids'])-supervised)
            h=int(hashlib.sha256((str(config['seed'])+fingerprint(record)).encode()).hexdigest()[:8],16)
            encoded.append(variants[0 if h%5==0 else 1])
        write_jsonl(root/'tokens'/f'{split}.jsonl',encoded)
        maxima[split]={'max':max(lengths),'records':len(encoded),'lengths':sorted(lengths)}
    # Cross-combining any observed condition with the longest completion must fit too.
    required=max(max(x['max'] for x in maxima.values()),prompt_max+completion_max)
    write_json(root/'length_report.json',{'splits':maxima,'required_context':required,
               'longest_completion':completion_max,'longest_prompt':prompt_max,'truncated_records':0})
    config['context_length']=choose_context(required)
    config['max_new_tokens']=completion_max
    tok.save_pretrained(root/'tokenizer')
    return config

def latest_checkpoint(target):
    """Newest checkpoint that has everything Trainer needs to resume (a kill mid-save leaves an incomplete one)."""
    need=('trainer_state.json','optimizer.pt','scheduler.pt','adapter_model.safetensors')
    good=[p for p in Path(target).glob('checkpoint-*') if p.is_dir() and all((p/n).exists() for n in need)]
    return max(good,key=lambda p:int(p.name.split('-')[-1]),default=None)

def train(config, smoke=False, resume=False):
    from model_runtime import load_model
    from unsloth import FastModel
    import torch
    import torch.distributed as dist
    from datasets import Dataset
    from trl import SFTConfig,SFTTrainer
    from transformers import TrainerCallback
    rank=int(os.environ.get('LOCAL_RANK','0')); world=int(os.environ.get('WORLD_SIZE','1'))
    if world!=2: raise RuntimeError('Launch with torchrun --standalone --nproc_per_node=2.')
    root=Path(config['run_dir']); target=root/('smoke' if smoke else 'checkpoints')
    model,tok=load_model(config,train=True)
    saved_tok_path=root/'tokenizer'
    # Ensure the model loader did not change the pinned tokenizer/template used for masks.
    from transformers import AutoTokenizer
    saved_tok=AutoTokenizer.from_pretrained(saved_tok_path,local_files_only=True)
    probe=[{'role':'user','content':'Generate synthetic data.'}]
    if tok.apply_chat_template(probe,tokenize=True,return_dict=False,add_generation_prompt=True,enable_thinking=False)!=saved_tok.apply_chat_template(probe,tokenize=True,return_dict=False,add_generation_prompt=True,enable_thinking=False):
        raise RuntimeError('Model loader changed the tokenizer/chat template after mask preparation.')
    examples=read_jsonl(root/'tokens/train.jsonl'); validation=read_jsonl(root/'tokens/validation.jsonl')
    # A full pass over Bitext's 3,900-row validation split at every quarter epoch
    # is slower than the training itself on T4s.  Use a deterministic held-out
    # subset for in-training checkpoint ranking; the separate selection and test
    # stages still measure the chosen adapter on untouched data.
    eval_limit=int(config.get('train_eval_examples',len(validation)))
    if eval_limit <= 0: raise ValueError('train_eval_examples must be positive.')
    if len(validation)>eval_limit:
        stride=max(1,len(validation)//eval_limit)
        validation=validation[::stride][:eval_limit]
    if smoke:
        # Exercise the worst-length batch, not merely a short first row.
        examples=sorted(examples,key=lambda x:len(x['input_ids']),reverse=True)[:2]*16
        validation=sorted(validation,key=lambda x:len(x['input_ids']),reverse=True)[:2]
    class Collator:
        def __call__(self,features):
            length=max(len(f['input_ids']) for f in features)
            if length>config['context_length']: raise RuntimeError('Refusing token truncation.')
            out={k:[] for k in ('input_ids','attention_mask','labels')}
            for f in features:
                pad=length-len(f['input_ids'])
                out['input_ids'].append(f['input_ids']+[tok.pad_token_id]*pad)
                out['attention_mask'].append([1]*len(f['input_ids'])+[0]*pad)
                out['labels'].append(f['labels']+[-100]*pad)
            return {k:torch.tensor(v,dtype=torch.long) for k,v in out.items()}
    def checkpoint_steps():
        return {int(p.name.split('-')[-1]) for p in target.glob('checkpoint-*') if p.is_dir() and (p/'adapter_model.safetensors').exists()}
    class Monitor(TrainerCallback):
        def __init__(self):
            self.best=math.inf; self.bad=0; self.scores={}; self.grad_checks=0; self.reason='epoch_limit'
            self.bad_grad_run=0; self.skipped_grad_steps=0
            state_path=target/'monitor_state.json'
            if resume and not smoke and state_path.exists():
                saved=read_json(state_path)
                self.scores={int(k):float(v) for k,v in saved['scores'].items()}
                self.best=min(self.scores.values(),default=math.inf); self.bad=int(saved.get('bad',0))
        def persist(self):
            write_json(target/'monitor_state.json',{'scores':{str(k):v for k,v in self.scores.items()},'bad':self.bad})
        def write_shortlist(self):
            present=checkpoint_steps()
            best=sorted((s for s in self.scores if s in present),key=lambda step:(self.scores[step],step))[:2]
            write_json(target/'shortlist.json',[{'path':str(target/f'checkpoint-{s}'),'step':s,'validation_loss':self.scores[s]} for s in best])
            return best
        def on_pre_optimizer_step(self,args,state,control,model=None,**kwargs):
            grads=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
            good=bool(grads) and all(bool(torch.isfinite(g).all()) for g in grads)
            flag=torch.tensor(int(good),device=f'cuda:{rank}')
            if dist.is_initialized(): dist.all_reduce(flag,op=dist.ReduceOp.MIN)
            if flag.item():
                self.bad_grad_run=0; self.grad_checks+=1; return
            # Smoke runs stay strict so a numerical problem is caught early. A long run tolerates an isolated
            # bad batch (the step is neutralised identically on every rank) but not a persistent failure.
            self.bad_grad_run+=1; self.skipped_grad_steps+=1
            if smoke or not grads or self.bad_grad_run>8: raise FloatingPointError('Missing or nonfinite adapter gradients.')
            for g in grads: g.zero_()
        def on_step_end(self,args,state,control,**kwargs):
            expired=torch.tensor(int(time.time()>=config['training_deadline'] and not smoke),device=f'cuda:{rank}')
            if dist.is_initialized(): dist.all_reduce(expired,op=dist.ReduceOp.MAX)
            if expired.item():
                self.reason='training_time_budget'; control.should_training_stop=True; control.should_evaluate=True; control.should_save=True
            return control
        def on_evaluate(self,args,state,control,metrics=None,**kwargs):
            loss=float(metrics['eval_loss'])
            if not math.isfinite(loss): raise FloatingPointError('Nonfinite validation loss.')
            self.scores[state.global_step]=loss
            if loss<self.best: self.best=loss; self.bad=0
            else: self.bad+=1
            if state.is_world_process_zero and not smoke: self.persist()
            patience=int(config.get('early_stopping_patience',4))   # 0 disables early stopping
            if patience and self.bad>=patience and not smoke: self.reason='early_stopping'; control.should_training_stop=True
            return control
        def on_save(self,args,state,control,**kwargs):
            if state.is_world_process_zero:
                best=self.write_shortlist()
                # Keep best two plus latest for recovery. Paths are confined to this run.
                keep=set(best+[state.global_step])
                for p in target.glob('checkpoint-*'):
                    if p.is_dir() and int(p.name.split('-')[-1]) not in keep:
                        if p.resolve().parent!=target.resolve(): raise RuntimeError('Unsafe checkpoint path')
                        shutil.rmtree(p)
            return control
    monitor=Monitor()
    interval=max(1,int(config.get('eval_every_steps',math.ceil(len(examples)/(16*4)))))
    save_interval=max(1,int(config.get('save_every_steps',interval)))
    requested_max_steps=int(config.get('max_train_steps',-1))
    args=SFTConfig(output_dir=str(target),per_device_train_batch_size=1,per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,num_train_epochs=config['epochs'],max_steps=2 if smoke else requested_max_steps,
        learning_rate=config['learning_rate'],warmup_ratio=.05,lr_scheduler_type='cosine',
        optim='adamw_torch',weight_decay=.01,max_grad_norm=1.0,fp16=True,bf16=False,
        gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},
        eval_strategy='steps',eval_steps=1 if smoke else interval,save_strategy='steps',save_steps=1 if smoke else save_interval,
        save_total_limit=None,logging_steps=1 if smoke else 10,report_to='none',seed=config['seed'],
        data_seed=config['seed'],packing=False,max_length=config['context_length'],
        dataset_kwargs={'skip_prepare_dataset':True},remove_unused_columns=False,
        ddp_find_unused_parameters=False,dataloader_num_workers=0,prediction_loss_only=True,
        load_best_model_at_end=False,save_safetensors=True)
    trainer=SFTTrainer(model=model,processing_class=tok,args=args,train_dataset=Dataset.from_list(examples),
        eval_dataset=Dataset.from_list(validation),data_collator=Collator(),callbacks=[monitor])
    # Compare cache modes on a short input; catches shared-KV regressions before long training.
    if smoke:
        model.eval(); tiny=torch.tensor([examples[0]['input_ids'][:16]],device=f'cuda:{rank}')
        with torch.no_grad():
            left=model(input_ids=tiny,use_cache=False).logits[:,-1].float()
            right=model(input_ids=tiny,use_cache=True).logits[:,-1].float()
        relative=float(torch.linalg.vector_norm(left-right)/torch.linalg.vector_norm(right).clamp_min(1e-9))
        if not math.isfinite(relative) or relative>.02: raise RuntimeError(f'Shared-KV cache parity failed: {relative}')
        model.train()
    else: relative=None
    torch.cuda.reset_peak_memory_stats()
    ckpt=latest_checkpoint(target) if (resume and not smoke) else None
    if resume and not smoke:
        if rank==0: print('RESUME: '+(f'continuing from {ckpt}' if ckpt else 'no complete checkpoint found; training from scratch'),flush=True)
        if ckpt and rank==0: monitor.write_shortlist()
    outcome=trainer.train(resume_from_checkpoint=str(ckpt) if ckpt else None)
    # Ensure a final scored checkpoint exists even for a fractional-epoch/time-budget stop.
    if trainer.state.global_step not in monitor.scores:
        metrics=trainer.evaluate(); monitor.scores[trainer.state.global_step]=float(metrics['eval_loss'])
        final=target/f'checkpoint-{trainer.state.global_step}'
        trainer.save_model(str(final))
        monitor.on_save(args,trainer.state,trainer.control)
    if trainer.is_world_process_zero() and not smoke: monitor.persist()
    if not smoke:
        trainer.save_model(str(root/'final_adapter'))   # the exact end-of-training weights, independent of the best-2 shortlist
        if trainer.is_world_process_zero(): tok.save_pretrained(root/'final_adapter')
    if not monitor.grad_checks: raise RuntimeError('Gradient checks were not invoked by this Trainer version.')
    if trainer.is_world_process_zero():
        report={'steps':trainer.state.global_step,'epoch':trainer.state.epoch,'stop_reason':monitor.reason,
            'training_examples':len(examples),'in_training_validation_examples':len(validation),
            'gradient_checks':monitor.grad_checks,'nonfinite_grad_steps_neutralised':monitor.skipped_grad_steps,'resumed_from':str(ckpt) if ckpt else None,'peak_memory_gb':torch.cuda.max_memory_allocated()/2**30,
            'cache_relative_error':relative,'metrics':outcome.metrics,'gpu':torch.cuda.get_device_name(rank)}
        write_json(root/('smoke_training.json' if smoke else 'training_report.json'),report)
        if smoke:
            adapter=root/'smoke_adapter'; trainer.save_model(str(adapter)); tok.save_pretrained(adapter)
    if dist.is_initialized(): dist.barrier()

if __name__=='__main__':
    parser=argparse.ArgumentParser(); parser.add_argument('--config',required=True); parser.add_argument('--smoke',action='store_true'); parser.add_argument('--resume',action='store_true')
    args=parser.parse_args(); train(read_json(args.config),args.smoke,args.resume)


Writing /kaggle/working/insurance_gemma_program/src/train.py


In [8]:
%%writefile /kaggle/working/insurance_gemma_program/src/worker.py
"""Fresh-process adapter verification and generation jobs."""
import argparse, time
from core import *

def run(config, job):
    from model_runtime import Generator
    profile=read_json(Path(config['run_dir'])/'profile.json')
    generator=Generator(config,profile,adapter=job.get('adapter'))
    if job.get('verify'):
        cond=sample_conditions(profile,1,config['seed'])[0]
        result=generator.one(cond,config['seed'],greedy=True,max_tokens=32,deadline=job.get('deadline'))
        if not result['raw'] or result['deadline_hit']: raise RuntimeError('Fresh-process generation failed or timed out.')
        write_json(job['output'],{'fresh_process_reload':True,'adapter':job.get('adapter'),'sample':result,
                                'note':'Reload/generation check; this is not a quality acceptance test.'})
        return
    for item in job['generations']:
        if time.time()>=job['deadline']: break
        generator.temperature=item['temperature']
        path=Path(item['output']); path.parent.mkdir(parents=True,exist_ok=True)
        path.write_text('',encoding='utf-8')
        generator.generate_synthetic(len(item['conditions']),item['seed'],mode=item.get('mode','raw'),
            conditions=item['conditions'],deadline=job['deadline'],trace_path=path)
        write_json(str(path)+'.status.json',generator.last_report)

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--config',required=True);p.add_argument('--job',required=True)
    args=p.parse_args();run(read_json(args.config),read_json(args.job))


Writing /kaggle/working/insurance_gemma_program/src/worker.py


In [9]:
%%writefile /kaggle/working/insurance_gemma_program/src/backup.py
"""Off-Kaggle backup/restore of a run directory to a PRIVATE Hugging Face repo. Never raises into the pipeline."""
import argparse, json, os, sys, time
from pathlib import Path

# Large and regenerable (raw downloads, token files) or useless after a restart (smoke artefacts, temp dirs).
IGNORE=['raw/*','tokens/*','smoke/*','smoke_adapter/*','**/tmp-*','**/tmp-*/*','*.part','__pycache__/*']

def api_and_repo(run_name):
    from huggingface_hub import HfApi
    api=HfApi(token=os.environ.get('HF_TOKEN'))
    user=api.whoami()['name']
    return api,f'{user}/{run_name}'

def fresh_checkpoint_patterns(run_dir,min_age=180):
    """Skip checkpoint folders touched in the last few minutes so a half-written file is never uploaded."""
    out=[]; now=time.time()
    for d in (Path(run_dir)/'checkpoints').glob('checkpoint-*'):
        if d.is_dir() and any(now-f.stat().st_mtime<min_age for f in d.rglob('*') if f.is_file()):
            out.append(f'checkpoints/{d.name}/*')
    return out

def push(run_dir,run_name):
    api,repo=api_and_repo(run_name)
    api.create_repo(repo,private=True,exist_ok=True)
    api.upload_folder(folder_path=str(run_dir),repo_id=repo,ignore_patterns=IGNORE+fresh_checkpoint_patterns(run_dir),
                      commit_message='backup '+time.strftime('%Y-%m-%d %H:%M:%S'))
    return repo

def pull(run_dir,run_name):
    from huggingface_hub import snapshot_download
    _,repo=api_and_repo(run_name)
    snapshot_download(repo,local_dir=str(run_dir),token=os.environ.get('HF_TOKEN'))
    return repo

def check(run_name):
    api,repo=api_and_repo(run_name)
    api.create_repo(repo,private=True,exist_ok=True)
    # Prove WRITE access (a read-only token passes whoami): upload a tiny marker file.
    api.upload_file(path_or_fileobj=b'ok',path_in_repo='backup_check.txt',repo_id=repo,commit_message='write check')
    return repo

if __name__=='__main__':
    p=argparse.ArgumentParser(); p.add_argument('cmd',choices=['check','push','pull'])
    p.add_argument('--run-dir'); p.add_argument('--run-name',required=True)
    a=p.parse_args()
    try:
        repo={'check':lambda:check(a.run_name),'push':lambda:push(a.run_dir,a.run_name),'pull':lambda:pull(a.run_dir,a.run_name)}[a.cmd]()
        print(json.dumps({'ok':True,'cmd':a.cmd,'repo':repo,'time':time.strftime('%H:%M:%S')}))
    except Exception as exc:
        print(json.dumps({'ok':False,'cmd':a.cmd,'error':f'{type(exc).__name__}: {str(exc)[:400]}'})); sys.exit(1)


Writing /kaggle/working/insurance_gemma_program/src/backup.py


In [10]:
%%writefile /kaggle/working/insurance_gemma_program/requirements.txt
# Direct dependencies pinned to mutually compatible declared ranges.
# Kaggle supplies the CUDA-enabled torch/xformers stack. Do not replace it with
# generic wheels: that can remove GPU support or mismatch the image's driver.
unsloth==2026.10.10
unsloth_zoo==2026.10.10
transformers==5.17.0
trl==1.15.0
peft==0.21.2
accelerate==1.15.0
bitsandbytes==0.50.2
datasets==4.8.5
huggingface_hub>=1.5.0,<2.0.0
sentence-transformers==6.2.0
# Use Kaggle's supported scientific stack when it meets these constraints;
# forcing older pins can break unrelated packages in the shared notebook image.
numpy>=1.26,<3
scipy>=1.11,<2
pandas>=2.2,<3
jsonschema==4.26.0
sentencepiece>=0.2,<0.3
protobuf>=5,<7
timm>=1.0,<2


Writing /kaggle/working/insurance_gemma_program/requirements.txt


## Kaggle GPU environment

Kaggle's Python 3.13 image cannot reliably create a `venv` because its `ensurepip` bundle may be absent.
Install the application packages into this disposable Kaggle session instead. Keep Kaggle's CUDA-enabled PyTorch,
torchvision, and xFormers packages; replacing them with generic PyPI wheels can remove GPU support or create an
incompatible CUDA stack. The pipeline's GPU smoke test verifies the installed environment before real training.
Installation may take several minutes. A failed install is logged; do not bypass its dependency checks.


In [11]:
gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True)
print(gpu_info)
assert len(gpu_info.strip().splitlines()) == 2, "Select GPU T4 x2 in Kaggle settings."
# Kaggle sessions are disposable. Using the active interpreter avoids the Python 3.13
# ensurepip/venv failure and lets child processes use the packages just installed.
PY = Path(sys.executable)
install_log = RUN_DIR / "installation.log"
commands = [
    [str(PY), "-m", "pip", "install", "--upgrade", "pip"],
    [str(PY), "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", "-r", str(PROGRAM / "requirements.txt")],
]
with install_log.open("w") as log:
    for cmd in commands:
        print("Installing/checking:", " ".join(cmd[3:]), flush=True)
        result = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)
        log.flush()
        if result.returncode:
            print(install_log.read_text()[-8000:])
            raise RuntimeError(f"Environment setup failed. Full log: {install_log}")
# `pip check` audits every unrelated package in Kaggle's shared image. Its existing
# conflicts (for example, a Colab-only package) do not tell us whether this pipeline works.
# Check exactly the runtime libraries and CUDA availability used by this notebook instead.
health = """
import torch, transformers, trl, peft, datasets, sentence_transformers, bitsandbytes, unsloth
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('transformers', transformers.__version__)
print('trl', trl.__version__)
print('peft', peft.__version__)
print('datasets', datasets.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device_count', torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() == 2
"""
probe = subprocess.check_output([str(PY), "-c", health], text=True, stderr=subprocess.STDOUT)
print(probe)
print("Environment installed. Full compatibility and numerical checks run next.")


Tesla T4, 580.178.04, 15360 MiB
Tesla T4, 580.178.04, 15360 MiB

Installing/checking: install --upgrade pip
Installing/checking: install --upgrade-strategy only-if-needed -r /kaggle/working/insurance_gemma_program/requirements.txt
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1666: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._

## Prepare, smoke-test, train, select, and evaluate

The pipeline profiles full records without truncation, checks the worst-length training batch on both GPUs,
verifies finite gradients and shared-KV cache behavior, and reloads the smoke adapter in a fresh process.
Training uses 70% of the configured work budget; validation generation, testing, and export use the remainder.
Metrics are computed on CPU to keep GPU processes independent. The final report distinguishes partial runs,
schema acceptance, constraints, copying, rejection rates, source overlap, and manual review.

Set `PREPARE_ONLY = True` to inspect downloads and splits without loading model weights. Use a new run name
for a second attempt; existing outputs are never overwritten by a new pipeline run.

The supervisor prints a heartbeat every 10 minutes, pushes the run folder to a private Hugging Face repo every `BACKUP_EVERY_MIN` minutes, and never raises at the end, so Kaggle always commits the outputs. With `train_until_done` training runs all epochs (ending only for a 25-minute reserve before `hours`), and selection/final generation share the time left; if that is not enough, resume later and the run goes straight to evaluation.


In [12]:
import signal
PREPARE_ONLY = False
BACKUP_SCRIPT = str(PROGRAM / "src/backup.py")

def backup_cmd(cmd):
    args = [str(PY), BACKUP_SCRIPT, cmd, "--run-name", RUN_NAME]
    if cmd != "check": args += ["--run-dir", str(RUN_DIR)]
    return args

def run_backup_blocking(cmd, timeout=900):
    try:
        out = subprocess.run(backup_cmd(cmd), capture_output=True, text=True, timeout=timeout)
        line = (out.stdout.strip().splitlines() or ["{}"])[-1]
        return json.loads(line)
    except Exception as exc:
        return {"ok": False, "error": f"{type(exc).__name__}: {exc}"}

# --- 1. backup must work BEFORE spending GPU time
backup_ok = False
if HF_TOKEN_PRESENT:
    res = run_backup_blocking("check", 300); backup_ok = bool(res.get("ok"))
    print("Backup check:", res)
if not backup_ok:
    msg = ("Off-Kaggle backup is NOT working (add a Kaggle secret HF_TOKEN with WRITE access and attach it to this notebook). "
           "Kaggle deletes /kaggle/working if a session is killed or the quota runs out.")
    if BACKUP_REQUIRED: raise RuntimeError(msg + " Set BACKUP_REQUIRED=False to run without protection.")
    print("WARNING:", msg)

# --- 2. optional resume of an interrupted run
if RESUME_RUN_NAME:
    assert RESUME_SOURCE, "Set RESUME_SOURCE to 'hf' or a path to the old run folder."
    if RESUME_SOURCE == "hf":
        res = run_backup_blocking("pull"); print("Restore:", res); assert res.get("ok"), "Restore from Hugging Face failed."
    else:
        shutil.copytree(RESUME_SOURCE, RUN_DIR, dirs_exist_ok=True)
    old = json.loads(CONFIG_PATH.read_text())
    old.update(hours=CONFIG["hours"], resume=True, run_dir=str(RUN_DIR)); old.pop("model_path", None)
    CONFIG.update(old); CONFIG_PATH.write_text(json.dumps(old, indent=2))
    print("Resuming run", RUN_NAME)

# --- 3. run the pipeline in the background and supervise it
cmd = [str(PY), "-u", str(PROGRAM / "src/run.py"), "--config", str(CONFIG_PATH)]
if PREPARE_ONLY:
    cmd.append("--prepare-only")
pipeline_log = RUN_DIR / "pipeline.log"
wall_limit = CONFIG["hours"] * 3600 + 900
began = time.monotonic(); last_beat = began; last_backup = began; backup_proc = None; pipeline_error = None
def tail_line(path):
    try:
        text = Path(path).read_bytes()[-4000:].decode("utf-8", "replace").replace("\r", "\n")
        lines = [l for l in text.splitlines() if l.strip()]
        return lines[-1][:300] if lines else ""
    except Exception: return ""
def start_backup():
    global backup_proc, last_backup
    if backup_ok and (backup_proc is None or backup_proc.poll() is not None):
        backup_proc = subprocess.Popen(backup_cmd("push"), stdout=open(RUN_DIR / "backup.log", "a"), stderr=subprocess.STDOUT)
        last_backup = time.monotonic()
try:
    with pipeline_log.open("w") as log:
        process = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        previous = ""
        while process.poll() is None:
            time.sleep(15)
            status_file = RUN_DIR / "session_status.json"
            current = status_file.read_text() if status_file.exists() else "Starting..."
            if current != previous:
                print(current, flush=True); previous = current
            now = time.monotonic()
            if now - last_beat > 600:   # heartbeat: last training/generation log line every 10 minutes
                last_beat = now
                print(f"[{(now-began)/3600:.2f}h] " + tail_line(RUN_DIR / "logs/train.log"), flush=True)
            if now - last_backup > BACKUP_EVERY_MIN * 60: start_backup()
            if now - began > wall_limit:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=180)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL)
                raise TimeoutError("Notebook watchdog stopped the controller. Logs and any exported adapter are preserved.")
        if process.returncode:
            pipeline_error = f"Pipeline stopped with exit code {process.returncode}. Inspect {RUN_DIR / 'failure_report.json'} and the logs."
except KeyboardInterrupt:
    if 'process' in globals() and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
    raise
except Exception as exc:
    pipeline_error = f"{type(exc).__name__}: {exc}"

# --- 4. final backup, then report. Deliberately does NOT raise: the notebook must finish normally so Kaggle commits /kaggle/working.
if backup_proc is not None and backup_proc.poll() is None:
    try: backup_proc.wait(timeout=900)
    except Exception: pass
if backup_ok:
    print("Final backup:", run_backup_blocking("push", 1800))
print(pipeline_log.read_text()[-8000:])
if pipeline_error:
    print("\n" + "!" * 78 + "\nPIPELINE DID NOT COMPLETE:", pipeline_error,
          "\nAny adapter that was saved is in adapter_DIAGNOSTIC_ONLY.zip / adapter_package_preliminary.zip.\n" + "!" * 78)


Backup check: {'ok': True, 'cmd': 'check', 'repo': 'adeelumar17/insuranceqa_20261031_194809', 'time': '19:49:47'}
{
  "status": "running",
  "stage": "tokenization_and_context_check",
  "elapsed_hours": 0.0025841371880637276,
  "resume": false
}
{
  "status": "running",
  "stage": "download_exact_base_weights",
  "elapsed_hours": 0.020855905413627626,
  "resume": false
}
{
  "status": "running",
  "stage": "two_gpu_smoke_training",
  "elapsed_hours": 0.04061239745881822,
  "resume": false
}
{
  "status": "running",
  "stage": "fresh_process_smoke_reload",
  "elapsed_hours": 0.12972158114115398,
  "resume": false
}
{
  "status": "running",
  "stage": "training",
  "elapsed_hours": 0.14216229134135777,
  "resume": false
}
[8.50h] 100%|█████████▉| 2400/2412 [8:20:33<02:10, 10.88s/it]Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/insuranceqa_20261031_194809/checkpoints/checkpoint-2400/tokenizer_config.json.
{
  "status": "running",
  "stage": "validation_generation",
  

## Results and downloads

A saved adapter is not a quality certificate. Check raw validity, novel-label/range errors, copying, the raw-versus-accepted distributions, and review at least the provided sample. The benchmark’s small sample cannot support strong generalization claims.


In [13]:
from IPython.display import display, FileLink
for name in ["session_status.json", "preparation_report.json", "training_report.json", "selection_report.json", "final_report.json"]:
    path = RUN_DIR / name
    if path.exists():
        value = json.loads(path.read_text())
        if name in ("session_status.json", "training_report.json"):
            print(name, json.dumps(value, indent=2)[:6000])
        display(FileLink(str(path)))
for relative in ["adapter_package.zip", "adapter_package_preliminary.zip", "adapter_DIAGNOSTIC_ONLY.zip", "final/synthetic.jsonl", "final/synthetic.csv", "final/manual_review.jsonl", "environment.lock.txt"]:
    path = RUN_DIR / relative
    if path.exists():
        display(FileLink(str(path)))


session_status.json {
  "status": "partial",
  "validation_acceptance_passed": false,
  "training_stop_reason": "epoch_limit",
  "package": "/kaggle/working/insuranceqa_20261031_194809/adapter_package.zip",
  "stage_errors": [],
  "quality_status": "Human review and interpretation required.",
  "elapsed_hours": 10.141658525268237
}


/kaggle/working/insuranceqa_20261031_194809/session_status.json

/kaggle/working/insuranceqa_20261031_194809/preparation_report.json

training_report.json {
  "steps": 2412,
  "epoch": 3.0,
  "stop_reason": "epoch_limit",
  "training_examples": 12856,
  "in_training_validation_examples": 400,
  "gradient_checks": 2412,
  "nonfinite_grad_steps_neutralised": 0,
  "resumed_from": null,
  "peak_memory_gb": 12.565269947052002,
  "cache_relative_error": null,
  "metrics": {
    "train_runtime": 30260.399,
    "train_samples_per_second": 1.275,
    "train_steps_per_second": 0.08,
    "total_flos": 3.477997735599145e+17,
    "train_loss": 3.356774225756897,
    "epoch": 3.0
  },
  "gpu": "Tesla T4"
}


/kaggle/working/insuranceqa_20261031_194809/training_report.json

/kaggle/working/insuranceqa_20261031_194809/selection_report.json

... (2 outputs left out)

/kaggle/working/insuranceqa_20261031_194809/adapter_package_preliminary.zip

/kaggle/working/insuranceqa_20261031_194809/final/synthetic.jsonl

/kaggle/working/insuranceqa_20261031_194809/final/manual_review.jsonl

/kaggle/working/insuranceqa_20261031_194809/environment.lock.txt

## Generate another batch

`generate_synthetic(n, seed, mode="validated")` loads the exported adapter in the isolated environment.
Validated mode returns only records passing the explicit checks; it can return fewer than requested after
five retries per request. Raw mode retains every attempt in the trace, including outputs that cannot be parsed.
Inspect `LAST_GENERATION_REPORT` and the trace file in both modes. Extra batches run after the timed experiment.


In [14]:
sys.path.insert(0, str(PROGRAM / "src"))
from core import read_json, read_jsonl, write_json, sample_conditions
def generate_synthetic(n, seed, mode="validated"):
    global LAST_GENERATION_REPORT
    if mode not in ("raw", "validated") or n < 0:
        raise ValueError("Use a nonnegative n and mode raw or validated.")
    exported = RUN_DIR / "export"
    settings = read_json(exported / "generation_config.json")
    profile = read_json(exported / "profile.json")
    tag = f"extra_{time.time_ns()}"
    output = RUN_DIR / f"{tag}.jsonl"
    job = {"adapter": str(exported / "adapter"), "deadline": time.time()+3600,
           "generations": [{"output": str(output), "conditions": sample_conditions(profile, n, seed),
                            "seed": seed, "temperature": settings["temperature"], "mode": mode}]}
    job_path = RUN_DIR / f"{tag}_job.json"
    write_json(job_path, job)
    env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
    subprocess.run([str(PY), str(PROGRAM / "src/worker.py"), "--config", str(CONFIG_PATH), "--job", str(job_path)], env=env, check=True)
    attempts = read_jsonl(output)
    LAST_GENERATION_REPORT = read_json(str(output)+".status.json")
    LAST_GENERATION_REPORT["trace_file"] = str(output)
    rows = [a["record"] for a in attempts if a["record"] is not None and (mode=="raw" or a["accepted"])]
    if CONFIG["dataset"] == "reliability":
        rows = [dict(r, id=f"synthetic-{seed}-{i:06d}") for i,r in enumerate(rows)]
    return rows

# Example (uncomment after completing the pipeline):
# records = generate_synthetic(10, seed=42)
# print(LAST_GENERATION_REPORT)
# print(json.dumps(records[:2], indent=2))


## References and boundaries

- [Gemma 4 E4B model](https://huggingface.co/google/gemma-4-E4B-it)
- [Unsloth Gemma 4 training](https://unsloth.ai/docs/models/gemma-4/train)
- [Unsloth DDP](https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth/ddp)
- [TRL completion loss](https://huggingface.co/docs/trl/sft_trainer)
- [InsuranceQA v2](https://github.com/shuzi/insuranceQA/tree/master/V2)
- [Bitext source and license](https://huggingface.co/datasets/bitext/Bitext-insurance-llm-chatbot-training-dataset)
- [Reliability benchmark](https://huggingface.co/datasets/pashas/insurance-ai-reliability-benchmark)

The code measures text/attribute similarity, not truth. Quantity checks cover explicit USD amounts, percentages,
days, months, ages, and structured numeric fields; they do not interpret every possible number in prose.
Matching those constraints does not guarantee that a record is in-distribution. No differential privacy guarantee is provided.
